In [2]:
import os

java_home = "/opt/homebrew/opt/openjdk@21"
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = f"{java_home}/bin:{os.environ['PATH']}"

print(os.environ["JAVA_HOME"])

/opt/homebrew/opt/openjdk@21


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[2]").appName("Dataops-copilot").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/25 23:33:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = spark.read.parquet("../data/yellow_tripdata_2026-01.parquet")

In [5]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [6]:
df.select("fare_amount", "trip_distance").printSchema()

root
 |-- fare_amount: double (nullable = true)
 |-- trip_distance: double (nullable = true)



In [7]:
df.select(
    "fare_amount",
    "trip_distance",
).summary("count", "min", "max", "mean").show()

+-------+-----------------+-----------------+
|summary|      fare_amount|    trip_distance|
+-------+-----------------+-----------------+
|  count|          3724889|          3724889|
|    min|          -2555.2|              0.0|
|    max|           2555.2|        269097.48|
|   mean|20.80425389321187|6.455646860884686|
+-------+-----------------+-----------------+



In [8]:
import pyspark.sql.functions as F

summary = df.selectExpr(
    "count(*) as row_count",
    "coalesce(avg(int(fare_amount IS NULL)), 0.0) as fare_null_rate",
    "coalesce(avg(int(trip_distance IS NULL)), 0.0) as trip_distance_null_rate ",
    "coalesce(avg(int(fare_amount < 0)), 0.0) as negative_fare_rate",
    "coalesce(avg(int(trip_distance <= 0)), 0.0) as invalid_trip_distance_rate",
)

summary.show()

+---------+--------------+-----------------------+--------------------+--------------------------+
|row_count|fare_null_rate|trip_distance_null_rate|  negative_fare_rate|invalid_trip_distance_rate|
+---------+--------------+-----------------------+--------------------+--------------------------+
|  3724889|           0.0|                    0.0|0.010594409658918695|       0.03375617367390008|
+---------+--------------+-----------------------+--------------------+--------------------------+



In [9]:
duplicates_all = df.groupBy(df.columns).count().filter(F.col("count") > 1)

In [10]:
duplicates_all = df.groupBy(df.columns).count().filter(F.col("count") > 1)
total_rows = df.count()
duplicate_count = total_rows - duplicates_all.count()
print(f"Total Duplicate Rows: {duplicate_count}")

Total Duplicate Rows: 3724889


In [ ]:
from pyspark.sql import DataFrame

from dataops_copilot.quality.models import QualityReport


def calculate_metrics(df: DataFrame) -> QualityReport:
    """Return the metrics as dict."""
    core_metrics = [
        F.count(F.lit(1)).alias("row_count"),
        F.coalesce(F.avg((F.col("fare_amount") < 0).cast("double")), F.lit(0.0)).alias(
            "negative_fare_rate"
        ),
        F.coalesce(F.avg((F.col("trip_distance") <= 0).cast("double")), F.lit(0.0)).alias(
            "invalid_trip_distance_rate"
        ),
    ]

    null_rate_aliases = {
        column_name: f"{column_name}_null_rate" for _, column_name in enumerate(df.columns)
    }

    null_rate_metrics = [
        F.coalesce(F.avg(F.col(column_name).isNull().cast("double")), F.lit(0.0)).alias(alias)
        for column_name, alias in null_rate_aliases.items()
    ]
    summary_row = df.agg(*(core_metrics + null_rate_metrics)).first()
    if summary_row is None:
        message = "Spark aggregation unexpectedly returned no metrics row."
        raise RuntimeError(message)

    metrics_summary = summary_row.asDict()
    row_count = metrics_summary["row_count"]
    if row_count > 0:
        distinct_count = df.distinct().count()
        duplicate_rate = (row_count - distinct_count) / row_count
    else:
        duplicate_rate = 0.0

    null_rates = {c: metrics_summary[f"{c}_null_rate"] for c in df.columns}
    return QualityReport(
        row_count=row_count,
        negative_fare_rate=metrics_summary["negative_fare_rate"],
        invalid_trip_distance_rate=metrics_summary["invalid_trip_distance_rate"],
        duplicate_rate=duplicate_rate,
        null_rates=null_rates,
    )



In [13]:
from dataops_copilot.quality.models import QualityReport

res = calculate_metrics(df)

In [17]:
res.dict()

/var/folders/4t/gzm07xdd5x30j9mntssjk4kw0000gn/T/ipykernel_3180/2071971495.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  res.dict()


{'row_count': 3724889,
 'null_rates': {'VendorID': 0.0,
  'tpep_pickup_datetime': 0.0,
  'tpep_dropoff_datetime': 0.0,
  'passenger_count': 0.2921048117138524,
  'trip_distance': 0.0,
  'RatecodeID': 0.2921048117138524,
  'store_and_fwd_flag': 0.2921048117138524,
  'PULocationID': 0.0,
  'DOLocationID': 0.0,
  'payment_type': 0.0,
  'fare_amount': 0.0,
  'extra': 0.0,
  'mta_tax': 0.0,
  'tip_amount': 0.0,
  'tolls_amount': 0.0,
  'improvement_surcharge': 0.0,
  'total_amount': 0.0,
  'congestion_surcharge': 0.2921048117138524,
  'Airport_fee': 0.2921048117138524,
  'cbd_congestion_fee': 0.0},
 'duplicate_rate': 0.0,
 'negative_fare_rate': 0.010594409658918695,
 'invalid_trip_distance_rate': 0.03375617367390008}

3724889